# Mahjong — oneshot (Claude)

1. Setup-Zelle ausführen (`LLM_BACKEND = "claude"`, timeout 1800s)
2. `run_generation()`
3. `run_full_evaluation(include_openspiel_compare=False)`

Setup: `LLM_MODEL = "opus"`, `LLM_EFFORT = "max"`. Artifacts: `outputs/mjh_claude_os.*`

Commit note: keep the `OK generation …s` line in the generation cell (wall-clock record).


In [4]:
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from pathlib import Path

_CHECKS_DIR = Path("checks").resolve()
if str(_CHECKS_DIR) not in sys.path:
    sys.path.insert(0, str(_CHECKS_DIR))
from common import (
    add_elapsed_to_result_output,
    parse_check_output_text,
    print_evaluation_summary,
)
from generation.config import normalize_backend, output_stem
from generation.llm_cli import build_llm_command, ensure_direct_llm_response, llm_failure_message, require_code_block_response, run_llm_subprocess

GAME = "mahjong"
OPEN_SPIEL_GAME = "none"
RUN_VARIANT = "oneshot"
LLM_BACKEND = "claude"
LLM_MODEL = "opus"
LLM_EFFORT = "max"  # comparable to gpt-5.5:xhigh
TIMEOUT_SECONDS = 3600 if RUN_VARIANT == "agentic" else 3600

USE_OPEN_SPIEL_BACKBONE = True
USE_IMPLEMENTATION_BRIEF = False

# Manual evaluation defaults: 100 rollouts for random-play checks.
# These checks are run by the user, not by coding agents during repo work.
ROLLOUTS = 100
MAX_STEPS = 300
CHECK_SEED = 1

INPUT_DIR = Path("inputs")
OUTPUT_DIR = Path("outputs")
PROMPT_PATH = Path("prompts/rulebook_to_python.txt")
BACKBONE_PATH = Path("prompts/open_spiel_backbone.md")
LLM_JUDGE_PROMPT_PATH = Path("prompts/llm_judge_review.md")
ACTION_LANGUAGE_PROMPT_PATH = Path("prompts/action_language_align.md")
ACTION_NORMALIZER_PATH = Path("checks/action_normalizer.py")
IMPLEMENTATION_BRIEF_PATH = OUTPUT_DIR / f"{GAME}_implementation_brief.md"

if RUN_VARIANT not in {"oneshot", "agentic"}:
    raise ValueError(f"Unsupported RUN_VARIANT: {RUN_VARIANT}")

RUN_STEM = output_stem(GAME, LLM_BACKEND, RUN_VARIANT)
JUDGE_BACKEND = normalize_backend(LLM_BACKEND)
CODE_PATH = OUTPUT_DIR / f"{RUN_STEM}.py"
RESPONSE_PATH = OUTPUT_DIR / f"{RUN_STEM}.md"
CHECK_LOG_PATH = OUTPUT_DIR / f"{RUN_STEM}_checks.txt"
JUDGE_REVIEW_PATH = OUTPUT_DIR / f"{RUN_STEM}_judge_{JUDGE_BACKEND}.md"
PRE_ALIGN_CODE_PATH = OUTPUT_DIR / f"{RUN_STEM}_pre_align.py"
ACTION_ALIGN_RESPONSE_PATH = OUTPUT_DIR / f"{RUN_STEM}_action_align.md"


def variant_paths(variant: str) -> dict[str, Path | str]:
    if variant not in {"oneshot", "agentic"}:
        raise ValueError(f"Unknown variant: {variant}")
    stem = output_stem(GAME, LLM_BACKEND, variant)
    judge_backend = normalize_backend(LLM_BACKEND)
    return {
        "variant": variant,
        "stem": stem,
        "code": OUTPUT_DIR / f"{stem}.py",
        "response": OUTPUT_DIR / f"{stem}.md",
        "check_log": OUTPUT_DIR / f"{stem}_checks.txt",
        "judge_review": OUTPUT_DIR / f"{stem}_judge_{judge_backend}.md",
        "pre_align": OUTPUT_DIR / f"{stem}_pre_align.py",
        "action_align_response": OUTPUT_DIR / f"{stem}_action_align.md",
    }


def find_rules_path() -> Path:
    input_dir = Path("inputs")
    if not input_dir.exists():
        raise FileNotFoundError("Missing inputs/ directory")

    rules_paths = sorted(
        path
        for path in input_dir.iterdir()
        if path.stem == "game_rules" and path.suffix.lower() in {".txt", ".pdf"}
    )
    if not rules_paths:
        raise FileNotFoundError("Keep exactly one of inputs/game_rules.txt or inputs/game_rules.pdf")
    if len(rules_paths) > 1:
        names = ", ".join(path.name for path in rules_paths)
        raise RuntimeError(f"Multiple game_rules files found; keep exactly one: {names}")
    return rules_paths[0]



def read_rules_text(rules_path: Path) -> str:
    if rules_path.suffix.lower() == ".txt":
        return rules_path.read_text(encoding="utf-8")

    if rules_path.suffix.lower() == ".pdf":
        try:
            from pypdf import PdfReader
        except ImportError as exc:
            raise ImportError("PDF rulebooks require pypdf; install requirements.txt in the boardbench Conda env") from exc

        reader = PdfReader(str(rules_path))
        return "\n\n".join(
            page_text.strip()
            for page in reader.pages
            for page_text in [page.extract_text() or ""]
            if page_text.strip()
        )

    raise ValueError(f"Unsupported rules file type: {rules_path.suffix}")


def get_pdf_renderer_path() -> str:
    renderer = shutil.which("pdftoppm") or shutil.which("pdftoppm.exe")
    if renderer:
        return renderer

    windows_fallback = Path.home() / "AppData/Local/Programs/MiKTeX/miktex/bin/x64/pdftoppm.exe"
    if windows_fallback.exists():
        return str(windows_fallback)

    raise FileNotFoundError(
        "Scanned/image PDFs require pdftoppm to render pages for pi image attachments"
    )


def render_pdf_pages(rules_path: Path) -> list[Path]:
    page_dir = INPUT_DIR / "rulebook_pages" / rules_path.stem
    page_dir.mkdir(parents=True, exist_ok=True)
    for old_page in page_dir.glob("page-*.png"):
        old_page.unlink()

    prefix = page_dir / "page"
    subprocess.run(
        [get_pdf_renderer_path(), "-png", "-r", "180", str(rules_path), str(prefix)],
        check=True,
        capture_output=True,
        text=True,
    )
    pages = sorted(page_dir.glob("page-*.png"))
    if not pages:
        raise RuntimeError(f"No PDF page images were rendered from {rules_path}")
    return pages


def build_rules_context() -> tuple[str, list[Path]]:
    rules_path = find_rules_path()
    rules_text = read_rules_text(rules_path)
    if rules_text.strip():
        return "Hier folgt die Spielanleitung:\n\n" + rules_text, []

    if rules_path.suffix.lower() == ".pdf":
        rendered_pages = render_pdf_pages(rules_path)
        page_list = "\n".join(f"- {path.as_posix()}" for path in rendered_pages)
        note = (
            "Die Spielanleitung ist eine bildbasierte PDF ohne extrahierbaren Text. "
            "Die gerenderten PDF-Seiten sind als Datei-Anhänge an diesen pi-Aufruf angehängt. "
            "Verwende ausschließlich diese Seiten als Regelquelle.\n\n"
            f"Gerenderte Seiten:\n{page_list}"
        )
        return note, rendered_pages

    raise ValueError(f"No usable rule text found in {rules_path}")

def optional_generation_inputs() -> list[tuple[str, Path, str]]:
    items: list[tuple[str, Path, str]] = []
    if USE_OPEN_SPIEL_BACKBONE and BACKBONE_PATH.exists():
        items.append(("OpenSpiel backbone", BACKBONE_PATH, BACKBONE_PATH.read_text(encoding="utf-8")))
    if USE_IMPLEMENTATION_BRIEF and IMPLEMENTATION_BRIEF_PATH.exists():
        items.append(
            (
                "Implementation brief",
                IMPLEMENTATION_BRIEF_PATH,
                IMPLEMENTATION_BRIEF_PATH.read_text(encoding="utf-8"),
            )
        )
    return items


def extract_code_block(text: str) -> str | None:
    match = re.search(r"```python\s*(.*?)```", text, re.IGNORECASE | re.DOTALL)
    if match is None:
        return None
    return match.group(1).strip() + "\n"



def build_one_shot_prompt() -> tuple[str, list[Path]]:
    if not PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing prompt file: {PROMPT_PATH}")

    rules_context, attachments = build_rules_context()
    prompt_parts = [PROMPT_PATH.read_text(encoding="utf-8")]
    for label, path, text in optional_generation_inputs():
        prompt_parts.append(f"# {label}: {path.as_posix()}\n\n{text}")
    return "\n\n".join(prompt_parts) + "\n\n" + rules_context, attachments




def build_agentic_prompt() -> tuple[str, list[Path]]:
    if not PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing prompt file: {PROMPT_PATH}")

    # Build attachments for image-only PDFs, but do not inline hidden check files.
    _rules_context, attachments = build_rules_context()
    lines = [
        "Work in the isolated temporary workspace prepared for this run.",
        "The only source-material directory is inputs/.",
        "Use only files in inputs/ as task context:",
        "- inputs/rulebook_to_python.txt",
        "- inputs/open_spiel_backbone.md, if present",
        "- inputs/implementation_brief.md, if present",
        "- inputs/game_rules.txt or inputs/game_rules.pdf",
        "- inputs/game_rules_extracted.txt, if present",
        "- inputs/rulebook_pages/*.png, if present or attached",
        "",
        "Do not read, copy, infer from, or run BoardBench benchmark checks.",
        "Do not access parent directories or repository files outside this isolated workspace.",
        "The evaluation checks must remain independent and unseen during generation.",
        "",
        "You may write the generated Python module under outputs/, inspect your own generated file,",
        "and run small self-contained Python syntax/import/logical smoke checks against that file.",
        "Do not use outside game knowledge or remembered rules.",
        "Use only the rulebook text or attached/rendered PDF page images as the game source of truth.",
        f"Write the final Python module to {CODE_PATH.as_posix()}.",
        "",
        "Your final response must contain:",
        "1. Open questions / assumptions",
        "2. one fenced python code block with the exact final file content",
    ]
    return "\n".join(lines), attachments


def copy_if_exists(source: Path, target: Path) -> bool:
    if not source.exists():
        return False
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    return True


def prepare_agentic_workspace(file_args: list[Path]) -> tuple[Path, list[Path]]:
    """Create a temporary generation workspace without BoardBench checks.

    The generator sees only copied source material in inputs/ plus its own
    outputs/ directory. This prevents leakage from benchmark checks while still
    allowing agentic self-review and syntax/logical smoke checks on its own code.
    """

    workspace = Path(tempfile.mkdtemp(prefix=f"boardbench_{GAME}_agentic_"))
    input_dir = workspace / "inputs"
    output_dir = workspace / "outputs"
    input_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    copy_if_exists(PROMPT_PATH, input_dir / "rulebook_to_python.txt")
    if USE_OPEN_SPIEL_BACKBONE:
        copy_if_exists(BACKBONE_PATH, input_dir / "open_spiel_backbone.md")
    if USE_IMPLEMENTATION_BRIEF:
        copy_if_exists(IMPLEMENTATION_BRIEF_PATH, input_dir / "implementation_brief.md")

    rules_path = find_rules_path()
    copy_if_exists(rules_path, input_dir / rules_path.name)
    rules_text = read_rules_text(rules_path)
    if rules_text.strip():
        (input_dir / "game_rules_extracted.txt").write_text(rules_text, encoding="utf-8")

    copied_args: list[Path] = []
    if file_args:
        page_dir = input_dir / "rulebook_pages"
        page_dir.mkdir(parents=True, exist_ok=True)
        for path in file_args:
            target = page_dir / path.name
            shutil.copy2(path, target)
            copied_args.append(target)

    return workspace, copied_args


def rewrite_attachment_paths(prompt_text: str, original_args: list[Path], copied_args: list[Path]) -> str:
    for original, copied in zip(original_args, copied_args):
        prompt_text = prompt_text.replace(original.as_posix(), copied.as_posix())
    return prompt_text


def copy_workspace_code(workspace: Path) -> bool:
    workspace_code_path = workspace / CODE_PATH
    if not workspace_code_path.exists():
        return False
    CODE_PATH.parent.mkdir(parents=True, exist_ok=True)
    CODE_PATH.write_text(workspace_code_path.read_text(encoding="utf-8"), encoding="utf-8")
    return True

def run_generation() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    prompt_text, file_args = build_one_shot_prompt() if RUN_VARIANT == "oneshot" else build_agentic_prompt()

    workspace: Path | None = None
    command_file_args = file_args
    cwd: Path | None = None
    if RUN_VARIANT == "agentic":
        workspace, command_file_args = prepare_agentic_workspace(file_args)
        prompt_text = rewrite_attachment_paths(prompt_text, file_args, command_file_args)
        cwd = workspace

    command = build_llm_command(
        LLM_BACKEND,
        LLM_MODEL,
        LLM_EFFORT,
        mode="agentic" if RUN_VARIANT == "agentic" else "oneshot",
        file_args=command_file_args,
        add_dirs=[cwd] if cwd is not None else None,
    )

    started = time.perf_counter()
    try:
        result = run_llm_subprocess(
            command,
            prompt_text=prompt_text,
            cwd=cwd,
            timeout=TIMEOUT_SECONDS,
        )

        RESPONSE_PATH.write_text(result.stdout or "", encoding="utf-8")
        if result.returncode != 0:
            raise RuntimeError(llm_failure_message(result, step="generation"))

        code = extract_code_block(result.stdout or "")
        if code is not None:
            CODE_PATH.write_text(code, encoding="utf-8")
        elif workspace is not None and copy_workspace_code(workspace):
            pass
        elif CODE_PATH.exists():
            pass
        else:
            raise RuntimeError("No fenced python block found in the LLM response and no code file was written")

        elapsed = time.perf_counter() - started
        print(f"OK generation {elapsed:.1f}s")
        return
    finally:
        if workspace is not None:
            shutil.rmtree(workspace, ignore_errors=True)

def check_command(
    code_path: Path,
    judge_path: Path,
    *,
    include_judge: bool = False,
    include_final: bool = False,
    only_checks: list[str] | None = None,
    no_summary: bool = False,
) -> list[str]:
    check_script = Path("checks/run_checks.py")
    if not check_script.exists():
        raise FileNotFoundError("Could not find checks/run_checks.py")

    cmd = [
        sys.executable,
        "-u",
        str(check_script),
        "--game",
        OPEN_SPIEL_GAME if include_final else GAME,
        "--code-path",
        str(code_path),
        "--rollouts",
        str(ROLLOUTS),
        "--max-steps",
        str(MAX_STEPS),
        "--seed",
        str(CHECK_SEED),
    ]
    if include_judge:
        cmd.append("--include-judge")
        cmd += ["--judge-path", str(judge_path)]
    if include_final:
        cmd.append("--include-final")
    if only_checks:
        for check_name in only_checks:
            cmd += ["--check", check_name]
    if no_summary:
        cmd.append("--no-summary")
    return cmd


STREAM_RESULT_PREFIXES = ("OK", "FAIL", "----")


def run_streaming_subprocess(
    cmd: list[str],
    *,
    log_path: Path | None = None,
    append_log: bool = False,
    stream_results: bool = True,
    output_transform=None,
) -> tuple[int, str]:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError("could not read subprocess output")

    lines: list[str] = []
    for line in process.stdout:
        lines.append(line)
        if stream_results and line.startswith(STREAM_RESULT_PREFIXES):
            print(line, end="", flush=True)

    returncode = process.wait()
    output = "".join(lines)
    if output_transform is not None:
        output = output_transform(output)
    if not stream_results:
        for line in output.splitlines(keepends=True):
            if line.startswith(STREAM_RESULT_PREFIXES):
                print(line, end="", flush=True)
    if log_path is not None:
        if append_log and log_path.exists():
            log_path.write_text(log_path.read_text(encoding="utf-8") + output, encoding="utf-8")
        else:
            log_path.write_text(output, encoding="utf-8")
    return returncode, output


def add_elapsed_to_first_result_line(output: str, extra_elapsed: float) -> str:
    return add_elapsed_to_result_output(output, extra_elapsed)


def parse_check_output(text: str) -> dict[str, int | float]:
    return parse_check_output_text(text)


def run_checks_for(
    *,
    code_path: Path,
    judge_path: Path,
    include_judge: bool = False,
    include_final: bool = False,
    only_checks: list[str] | None = None,
    log_path: Path | None = None,
    label: str | None = None,
    append_log: bool = False,
    no_summary: bool = False,
    raise_on_failure: bool = True,
    extra_elapsed: float = 0.0,
) -> tuple[int, dict[str, int | float]]:
    cmd = check_command(
        code_path=code_path,
        judge_path=judge_path,
        include_judge=include_judge,
        include_final=include_final,
        only_checks=only_checks,
        no_summary=no_summary,
    )
    returncode, output = run_streaming_subprocess(
        cmd,
        log_path=log_path,
        append_log=append_log,
        stream_results=extra_elapsed <= 0,
        output_transform=(lambda text: add_elapsed_to_first_result_line(text, extra_elapsed)) if extra_elapsed > 0 else None,
    )
    stats = parse_check_output(output)
    if raise_on_failure and returncode != 0:
        raise RuntimeError(f"checks failed with exit code {returncode}")
    return returncode, stats



def run_pair_action_compare(
    *,
    left_code_path: Path,
    right_code_path: Path,
    left_label: str = "oneshot",
    right_label: str = "agentic",
) -> None:
    compare_script = Path("checks/compare_pair.py")
    if not compare_script.exists():
        raise FileNotFoundError("Could not find checks/compare_pair.py")

    cmd = [
        sys.executable,
        "-u",
        str(compare_script),
        "--game",
        GAME,
        "--left-code-path",
        str(left_code_path),
        "--right-code-path",
        str(right_code_path),
        "--left-label",
        left_label,
        "--right-label",
        right_label,
        "--rollouts",
        str(ROLLOUTS),
        "--max-steps",
        str(MAX_STEPS),
        "--seed",
        str(CHECK_SEED),
    ]
    pair_log_path = OUTPUT_DIR / f"{GAME}_pair_action_compare.txt"
    returncode, _output = run_streaming_subprocess(cmd, log_path=pair_log_path)
    if returncode != 0:
        raise RuntimeError(f"pair action-language comparison failed with exit code {returncode}")



def build_judge_packet_for(
    *,
    code_path: Path,
    check_log_path: Path,
    output_path: Path,
    judge_review_path: Path,
    variant: str,
) -> Path:
    if not code_path.exists():
        raise FileNotFoundError(f"Missing generated code: {code_path}")
    if not PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing prompt file: {PROMPT_PATH}")
    if not LLM_JUDGE_PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing judge prompt file: {LLM_JUDGE_PROMPT_PATH}")

    rules_path = find_rules_path()
    sections = [
        "# BoardBench judge packet",
        f"- game: {GAME}",
        f"- OpenSpiel reference: {OPEN_SPIEL_GAME}",
        f"- variant: {variant}",
        f"- generated code: {code_path.as_posix()}",
        f"- expected judge reply path: {judge_review_path.as_posix()}",
        "",
        "## Judge prompt",
        LLM_JUDGE_PROMPT_PATH.read_text(encoding="utf-8"),
        "",
        f"## Generation prompt ({PROMPT_PATH.as_posix()})",
        PROMPT_PATH.read_text(encoding="utf-8"),
    ]

    for label, path, text in optional_generation_inputs():
        sections += ["", f"## {label} ({path.as_posix()})", text]

    rules_text = read_rules_text(rules_path)
    if rules_text.strip():
        sections += ["", f"## Rule text ({rules_path.as_posix()})", rules_text]
    elif rules_path.suffix.lower() == ".pdf":
        page_paths = render_pdf_pages(rules_path)
        sections += [
            "",
            f"## Rulebook PDF ({rules_path.as_posix()})",
            "The PDF has no extractable text. Use these rendered page images as the rulebook source:",
        ]
        sections += [f"![{path.name}]({path.as_posix()})" for path in page_paths]
    else:
        sections += ["", "## Rule text", "_No extractable rule text found._"]

    sections += [
        "",
        f"## Generated code ({code_path.as_posix()})",
        "```python\n" + code_path.read_text(encoding="utf-8") + "```",
    ]

    # The LLM judge is a rulebook-vs-code scoring step, not a rerun of the
    # deterministic BoardBench checks. Keep check logs out of the packet so the
    # judge does not simply repeat mechanical failures.
    output_path.write_text("\n\n".join(sections), encoding="utf-8")
    return output_path




def run_judge(
    *,
    code_path: Path | None = None,
    packet_path: Path | None = None,
    review_path: Path | None = None,
    check_log_path: Path | None = None,
    variant: str | None = None,
) -> float:
    started = time.perf_counter()
    code_path = code_path or CODE_PATH
    review_path = review_path or JUDGE_REVIEW_PATH
    check_log_path = check_log_path or CHECK_LOG_PATH
    variant = variant or RUN_VARIANT

    if review_path.exists() and review_path.read_text(encoding="utf-8").strip():
        return 0.0

    created_packet = False
    if packet_path is None:
        fd, raw_path = tempfile.mkstemp(prefix=f"judge_packet_{RUN_STEM}_", suffix=".md")
        os.close(fd)
        packet_path = Path(raw_path)
        created_packet = True

    try:
        build_judge_packet_for(
            code_path=code_path,
            check_log_path=check_log_path,
            output_path=packet_path,
            judge_review_path=review_path,
            variant=variant,
        )

        judge_attachments = []
        rules_path = find_rules_path()
        if rules_path.suffix.lower() == ".pdf" and not read_rules_text(rules_path).strip():
            judge_attachments = render_pdf_pages(rules_path)

        command = build_llm_command(
            LLM_BACKEND,
            LLM_MODEL,
            LLM_EFFORT,
            mode="judge",
            file_args=judge_attachments,
            add_dirs=[INPUT_DIR] if judge_attachments else None,
        )
        prompt_text = packet_path.read_text(encoding="utf-8")
        result = run_llm_subprocess(
            command,
            prompt_text=prompt_text,
            timeout=TIMEOUT_SECONDS,
        )
        review_text = ensure_direct_llm_response(result, step="judge")
        review_path.write_text(review_text, encoding="utf-8")
        return time.perf_counter() - started
    finally:
        if created_packet:
            try:
                packet_path.unlink(missing_ok=True)
            except OSError:
                pass







def run_full_evaluation_for(
    *,
    code_path: Path,
    judge_path: Path,
    check_log_path: Path,
    label: str,
    judge_packet_path: Path | None = None,
    include_llm_judge: bool = True,
    include_openspiel_compare: bool = True,
) -> None:
    phase_stats: list[dict[str, int | float]] = []
    phase_labels: list[str] = ["base checks"]
    failed_phases: list[str] = []

    returncode, stats = run_checks_for(
        code_path=code_path,
        judge_path=judge_path,
        include_judge=False,
        include_final=False,
        log_path=check_log_path,
        label=label,
        no_summary=True,
        raise_on_failure=False,
    )
    phase_stats.append(stats)
    if returncode != 0:
        failed_phases.append("base checks")

    if include_llm_judge:
        judge_elapsed = run_judge(
            code_path=code_path,
            packet_path=judge_packet_path,
            review_path=judge_path,
            check_log_path=check_log_path,
            variant=label,
        )
        returncode, stats = run_checks_for(
            code_path=code_path,
            judge_path=judge_path,
            include_judge=True,
            include_final=False,
            only_checks=["90_llm_judge"],
            log_path=check_log_path,
            label=label,
            append_log=True,
            no_summary=True,
            raise_on_failure=False,
            extra_elapsed=judge_elapsed,
        )
        phase_stats.append(stats)
        phase_labels.append("llm judge")
        if returncode != 0:
            failed_phases.append("llm judge")

    if include_openspiel_compare:
        returncode, stats = run_openspiel_workflow_for(
            code_path=code_path,
            judge_path=judge_path,
            log_path=check_log_path,
            append_log=True,
            no_summary=True,
            raise_on_failure=False,
        )
        phase_stats.append(stats)
        phase_labels.append("openspiel compare")
        if returncode != 0:
            failed_phases.append("openspiel compare")

    print_evaluation_summary(
        phase_stats,
        phase_labels=phase_labels,
        log_path=check_log_path,
    )
    if failed_phases:
        raise RuntimeError("evaluation failed in: " + ", ".join(failed_phases))


def run_full_evaluation(
    *,
    include_llm_judge: bool = True,
    include_openspiel_compare: bool = True,
) -> None:
    run_full_evaluation_for(
        code_path=CODE_PATH,
        judge_path=JUDGE_REVIEW_PATH,
        check_log_path=CHECK_LOG_PATH,
        judge_packet_path=None,
        label=RUN_VARIANT,
        include_llm_judge=include_llm_judge,
        include_openspiel_compare=include_openspiel_compare,
    )

def run_openspiel_workflow_for(
    *,
    code_path: Path,
    judge_path: Path,
    log_path: Path | None = None,
    append_log: bool = False,
    no_summary: bool = False,
    raise_on_failure: bool = True,
) -> tuple[int, dict[str, int | float]]:
    align_elapsed = run_action_language_align(code_path=code_path)
    return run_checks_for(
        code_path=code_path,
        judge_path=judge_path,
        include_judge=False,
        include_final=True,
        only_checks=["99_openspiel_compare"],
        log_path=log_path,
        append_log=append_log,
        no_summary=no_summary,
        raise_on_failure=raise_on_failure,
        extra_elapsed=align_elapsed,
    )

def run_action_language_align(
    *,
    code_path: Path | None = None,
    response_path: Path | None = None,
    pre_align_path: Path | None = None,
) -> float:
    started = time.perf_counter()
    code_path = code_path or CODE_PATH
    response_path = response_path or OUTPUT_DIR / f"{code_path.stem}_action_align.md"
    pre_align_path = pre_align_path or OUTPUT_DIR / f"{code_path.stem}_pre_align.py"
    normalizer_path = Path("checks/action_normalizer.py")

    if not code_path.exists():
        raise FileNotFoundError(f"Missing generated code: {code_path}")
    if not ACTION_LANGUAGE_PROMPT_PATH.exists():
        raise FileNotFoundError(f"Missing action-language prompt: {ACTION_LANGUAGE_PROMPT_PATH}")
    if not normalizer_path.exists():
        raise FileNotFoundError(f"Missing action normalizer: {normalizer_path}")

    shutil.copy2(code_path, pre_align_path)
    prompt_text = "\n\n".join(
        [
            ACTION_LANGUAGE_PROMPT_PATH.read_text(encoding="utf-8"),
            "## BoardBench action normalizer",
            "```python",
            normalizer_path.read_text(encoding="utf-8"),
            "```",
            f"## Generated code ({code_path.as_posix()})",
            "```python",
            code_path.read_text(encoding="utf-8"),
            "```",
        ]
    )

    result = run_llm_subprocess(
        build_llm_command(
            LLM_BACKEND,
            LLM_MODEL,
            LLM_EFFORT,
            mode="align",
        ),
        prompt_text=prompt_text,
        timeout=TIMEOUT_SECONDS,
    )
    response_path.write_text(result.stdout or "", encoding="utf-8")
    code = require_code_block_response(result, step="action-language align")
    code_path.write_text(code, encoding="utf-8")
    return time.perf_counter() - started




## One-shot generation

`run_generation()` with `LLM_BACKEND = "claude"` (`claude -p`, subscription login).


In [5]:
run_generation()


OK generation 1704.0s


## Full evaluation

Run after generation.

One cell runs the full pipeline: base checks, LLM judge, judge check, and optional OpenSpiel align+compare.


In [6]:
INCLUDE_LLM_JUDGE = True
INCLUDE_OPENSPIEL_COMPARE = False

run_full_evaluation(
    include_llm_judge=INCLUDE_LLM_JUDGE,
    include_openspiel_compare=INCLUDE_OPENSPIEL_COMPARE,
)


OK   01_result_file                     1/1 score=1.000    0.00s
OK   02_python_syntax                   1/1 score=1.000    0.01s
OK   03_startable_game                  1/1 score=1.000    0.01s
OK   04_required_api                    8/8 score=1.000    0.00s
OK   05_random_rollouts             100/100 score=1.000    3.16s
OK   06_action_language       405999/405999 score=1.000    2.37s
OK   90_llm_judge                    70/100 score=0.700 1023.31s
---- phase base checks                  6/6 score=1.000    5.55s
---- phase llm judge                    1/1 score=0.700 1023.31s
---- summary                            7/7 score=0.912 1028.86s
